[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_Summarization.ipynb)

# Benchmark: Summarization

Scores a pretrained summarization model's ROUGE against gold-labeled data, using
`sparknlp.benchmark.Benchmark.evaluate(..., task="summarization")`.

**Dataset**: a small hand-written sample of short news-style paragraphs, each with a
one-sentence reference summary -- matching the style of
[XSum](https://huggingface.co/datasets/EdinburghNLP/xsum) (single, highly abstractive
summaries), which is what the model below was actually fine-tuned on. For a larger benchmark,
pull a real sample from the XSum dataset directly.

**Model**: `BartTransformer.pretrained("distilbart_xsum_12_6")`, a DistilBART model fine-tuned
on XSum.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-09-18 02:07:51--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-09-18 02:07:51--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’

-                   100%[===================>]   1.45K  --.-KB/s    in 0s      

2026-09-18 02:07:51 (30.7 MB/s) - written to stdout [1483/1483]

Installing PySpa

In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.20+8-1~24.04_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.20+8-1~24.04) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.20+8-1~24.04_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.20+8-1~24.04) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.20+8-1~24.04) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.024s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.025s][warning]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import BartTransformer
from pyspark.ml import Pipeline
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
rows = [
    ("Scientists at a European research institute announced Thursday that they had successfully "
     "sequenced the genome of a rare Alpine flower previously thought to be extinct in the wild. "
     "The five-year project involved collecting samples from remote mountain valleys and could "
     "help conservationists breed the plant back into its native habitat.",
     "Scientists have sequenced the genome of a rare Alpine flower thought to be extinct."),
    ("The city council voted unanimously on Tuesday to approve funding for a new public transit "
     "line connecting the northern suburbs to the downtown business district. Construction is "
     "expected to begin next spring and take approximately eighteen months to complete, with "
     "officials saying it will reduce commute times by up to thirty minutes.",
     "The city council approved funding for a new transit line linking the suburbs to downtown."),
    ("A local bakery that has served the same neighborhood for over sixty years announced it "
     "will close its doors at the end of the month after the owner decided to retire. Longtime "
     "customers gathered outside the shop this week to share memories and thank the family for "
     "decades of fresh bread and pastries.",
     "A neighborhood bakery is closing after sixty years when its owner retires."),
    ("Researchers published a study this week showing that a newly developed battery material "
     "could charge electric vehicles in under ten minutes while retaining most of its capacity "
     "after thousands of charge cycles. The team says commercial production is still several "
     "years away but early tests have exceeded expectations.",
     "Researchers developed a new battery material that could charge electric vehicles in "
     "under ten minutes."),
    ("A small coastal town saw record numbers of visitors this summer after a viral social media "
     "video showcased its beaches and seafood restaurants. Local officials say the tourism boom "
     "has strained parking and water supplies, prompting the town to consider new visitor limits "
     "for next year.",
     "A coastal town saw a tourism boom after a viral video, straining local resources."),
    ("The national weather service issued a warning ahead of an expected heat wave that could "
     "push temperatures above forty degrees Celsius in several regions this weekend. Health "
     "officials are urging residents to stay hydrated, check on elderly neighbors, and avoid "
     "outdoor activity during peak afternoon hours.",
     "Officials issued a heat wave warning and urged residents to take precautions."),
    ("A university robotics team won first place at an international competition after building "
     "a search-and-rescue robot capable of navigating rubble and detecting survivors using "
     "thermal cameras. The team plans to publish their design so other groups can build on "
     "the technology.",
     "A university robotics team won an international competition with a search-and-rescue "
     "robot."),
    ("Airlines are reporting a sharp increase in bookings for winter holiday travel compared to "
     "last year, with several major carriers adding extra flights on popular routes. Analysts "
     "attribute the rise to pent-up demand and falling ticket prices earlier in the season.",
     "Airlines are seeing a sharp rise in winter holiday bookings compared to last year."),
]

gold_data = spark.createDataFrame(rows, ["text", "label"]).repartition(1)

> **Two notes on running BART safely under Spark:**
> - **No `.setTask(...)` prefix.** `BartTransformer`'s docstring examples borrow a
>   `.setTask("summarize:")` pattern from T5 -- but that prefix convention is a T5-ism (one
>   text-to-text model trained on many tasks, disambiguated by a prefix). This checkpoint is
>   BART, already fine-tuned only on XSum summarization, and feeding it an unexpected prefix
>   token breaks its decoder.
> - **`.repartition(1)` on the input.** Multiple Spark tasks can end up concurrently driving the
>   same loaded TensorFlow graph, which surfaces as sporadic, differently-shaped
>   `Incompatible shapes` errors. `setBatchSize(1)` alone doesn't prevent this (it controls how
>   many annotations one TF call processes, not how many Spark tasks run concurrently) --
>   repartitioning to a single partition, so one task handles all rows sequentially, does.

## 2. Build the pipeline

In [11]:
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
summarizer = BartTransformer.pretrained("distilbart_xsum_12_6") \
    .setInputCols(["document"]) \
    .setMaxOutputLength(60) \
    .setBatchSize(1) \
    .setOutputCol("summary")

pipeline = Pipeline(stages=[document_assembler, summarizer])
pipeline_model = pipeline.fit(gold_data)

distilbart_xsum_12_6 download started this may take some time.
Approximate size to download 699.7 MB
[OK!]

> **Note: `predicted_col` disambiguates same-typed columns.** Both `document` (the raw
> input, echoed by `DocumentAssembler`) and `summary` (the model's actual output) are annotator
> type `document` -- auto-detection can't tell them apart, so we pass `predicted_col`
> explicitly, the same disambiguation used in the `SpellCheck` notebook.

## 3. Run the benchmark

In [14]:
# DocumentAssembler and BartTransformer both emit annotatorType "document"; Benchmark scores the
# last one (the summary) and names it in the report.
report = Benchmark.evaluate(pipeline_model, gold_data, task="summarization", label_col="label")
print(report)

summarization accuracy (n=8, scored: summary): rouge1_f1=0.2825, rouge1_precision=0.2214, rouge1_recall=0.4045, rouge2_f1=0.1172, rouge2_precision=0.0921, rouge2_recall=0.1646, rougeL_f1=0.2358, rougeL_precision=0.1861, rougeL_recall=0.3315